## Часть 4Е. Метрики качества: пишем их сами

Модель обучена – но насколько она хороша? Ответ дают **метрики**. В этой части вы напишете их с нуля
и поймете, что скрывается за `accuracy_score`, `precision_score`, `roc_auc_score` и `log_loss`.

**Все начинается с матрицы ошибок.** Класс 1 – «положительный» (спам, болезнь), класс 0 – «отрицательный».
Каждый ответ модели попадает ровно в одну из четырех клеток:

| | модель сказала **1** | модель сказала **0** |
|---|---|---|
| **на самом деле 1** | **TP** – верно нашли | **FN** – пропустили |
| **на самом деле 0** | **FP** – ложная тревога | **TN** – верно отвергли |

Запомнить просто: вторая буква (**P**/**N**) – что *сказала модель*, первая (**T**/**F**) – *права ли она*.

**План части.**

| Задание | Что пишем | Главная формула |
|---|---|---|
| Е1 | матрица ошибок | счетчики TP, FP, FN, TN |
| Е2 | accuracy, precision, recall, specificity | $\frac{TP}{TP+FP}$, $\frac{TP}{TP+FN}$ |
| Е3 | F-мера | $F_1 = \frac{2PR}{P+R}$ |
| Е4 | порог классификации | $\hat y = [\,p \ge t\,]$ |
| Е5 | ROC-кривая и AUC | площадь трапециями |
| Е6 | функции потерь | log loss, Brier |
| Е7 | несколько классов | macro- и micro-усреднение |
| Е8 | регрессия | MAE, MSE, RMSE, $R^2$ |

> **Правило части.** Внутри ваших функций – только списки, словари, циклы, `sum`, `len`, `zip` и `math`.
> Никаких numpy и sklearn. Sklearn используется **только** в готовых ячейках – чтобы получить ответы моделей и сверить ваши числа.
>
> После каждого задания есть ячейка **«Проверка»**. Все строки начинаются с `OK` – можно идти дальше.

In [ ]:
# Ячейка готова. Библиотеки, данные и функция проверки.
import math
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_wine, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics as skm          # только для сверки
RANDOM_STATE = 42

def check(name, compute, expected, tol=1e-4):
    # compute – функция без аргументов (lambda), expected – ожидаемый ответ
    try:
        got = compute()
    except Exception as e:
        print(f'НЕТ  {name}: ошибка {type(e).__name__}: {e}')
        return
    if isinstance(expected, float):
        ok = isinstance(got, (int, float)) and abs(got - expected) <= tol
        shown = round(got, 4) if isinstance(got, (int, float)) else got
        print(f"{'OK ' if ok else 'НЕТ'}  {name}: получено {shown}, ожидается {round(expected, 4)}")
    else:
        ok = got == expected
        print(f"{'OK ' if ok else 'НЕТ'}  {name}: получено {got}, ожидается {expected}")

# 1) Учебный пример: 10 писем. 1 – спам, 0 – обычное письмо.
#    p_toy – вероятность «спам», которую выдала модель. При пороге 0.5 получаем pred_toy.
y_toy = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
p_toy = [0.9, 0.8, 0.6, 0.3, 0.7, 0.4, 0.35, 0.2, 0.1, 0.05]
pred_toy = [1 if p >= 0.5 else 0 for p in p_toy]

# 2) Спам-фильтр из статьи ODS: 100 обычных писем и 10 спамных.
y_spam = [0] * 100 + [1] * 10
pred_model = [0] * 90 + [1] * 10 + [1] * 5 + [0] * 5   # честная модель
pred_lazy = [0] * 110                                  # «ленивая» модель: всегда «не спам»

# 3) Диагностика опухоли (Breast Cancer): 1 – злокачественная, 0 – доброкачественная.
bc = load_breast_cancer()
yb = [1 - int(t) for t in bc.target]
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(bc.data, yb, test_size=0.3, stratify=yb, random_state=RANDOM_STATE)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xb_tr, yb_tr)
tree2 = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE).fit(Xb_tr, yb_tr)
yb_te = [int(t) for t in yb_te]
pb_logreg = logreg.predict_proba(Xb_te)[:, 1].tolist()     # вероятности «злокачественная»
pb_tree = tree2.predict_proba(Xb_te)[:, 1].tolist()
predb = [1 if p >= 0.5 else 0 for p in pb_logreg]

# 4) Три сорта вина (Wine) и дерево глубины 3 – как в части 4Д.
wine = load_wine()
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(wine.data, wine.target, test_size=0.3, stratify=wine.target, random_state=RANDOM_STATE)
wine_tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(Xw_tr, yw_tr)
yw_te = [int(t) for t in yw_te]
predw = [int(p) for p in wine_tree.predict(Xw_te)]

# 5) Регрессия. Учебный пример: прогноз температуры на 5 дней, °C.
t_true = [3, 5, 2, 8, 7]
t_pred = [2.5, 5, 4, 7, 7.5]
#    Diabetes: линейная регрессия предсказывает прогресс болезни через год.
db = load_diabetes()
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(db.data, db.target, test_size=0.3, random_state=RANDOM_STATE)
linreg = LinearRegression().fit(Xd_tr, yd_tr)
yd_te = yd_te.tolist()
predd = linreg.predict(Xd_te).tolist()

print('Письма:', len(y_toy), ' Опухоли (тест):', len(yb_te), ' Вино (тест):', len(yw_te), ' Diabetes (тест):', len(yd_te))

### Задание Е1. Матрица ошибок

Напишите `confusion_matrix(y_true, y_pred)` – она возвращает словарь `{'TP': ..., 'FP': ..., 'FN': ..., 'TN': ...}`.

| правда `t` | ответ `p` | клетка |
|---|---|---|
| 1 | 1 | TP |
| 0 | 1 | FP |
| 1 | 0 | FN |
| 0 | 0 | TN |

*Подсказка:* пройдите по парам `zip(y_true, y_pred)` и увеличьте нужный счетчик.

In [ ]:
# ЗАДАНИЕ Е1
def confusion_matrix(y_true, y_pred):
    # ваш код
    pass

In [ ]:
# Проверка Е1 (ячейка готова)
print('pred_toy =', pred_toy)
check('письма', lambda: confusion_matrix(y_toy, pred_toy), {'TP': 3, 'FP': 1, 'FN': 1, 'TN': 5})
check('спам-фильтр', lambda: confusion_matrix(y_spam, pred_model), {'TP': 5, 'FP': 10, 'FN': 5, 'TN': 90})
tn, fp, fn, tp = skm.confusion_matrix(yb_te, predb).ravel()
check('опухоли vs sklearn', lambda: confusion_matrix(yb_te, predb), {'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn)})

**Вопрос Е1.** Какое письмо из учебного примера стало ложной тревогой (FP), а какое – пропуском (FN)? Посмотрите на `p_toy`.
В задаче «найти опухоль» какая ошибка страшнее – FP или FN? А в спам-фильтре?

*Ваш ответ:*

### Задание Е2. Четыре главные доли

| Метрика | Формула | Вопрос, на который она отвечает |
|---|---|---|
| **Accuracy** (доля верных) | $\dfrac{TP + TN}{TP + TN + FP + FN}$ | Как часто модель права? |
| **Precision** (точность) | $\dfrac{TP}{TP + FP}$ | Из тех, кого модель назвала «1», сколько на самом деле «1»? |
| **Recall** (полнота, чувствительность) | $\dfrac{TP}{TP + FN}$ | Из всех настоящих «1», сколько модель нашла? |
| **Specificity** (специфичность) | $\dfrac{TN}{TN + FP}$ | Из всех настоящих «0», сколько модель верно отвергла? |

Все четыре – числа от 0 до 1, чем больше, тем лучше. Внутри используйте `confusion_matrix` из Е1.
Если знаменатель равен 0 (например, модель ни разу не сказала «1»), верните `0.0` – для этого дана функция `safe_div`.

In [ ]:
# ЗАДАНИЕ Е2
def safe_div(a, b):
    return a / b if b != 0 else 0.0

def accuracy(y_true, y_pred):
    # ваш код
    pass

def precision(y_true, y_pred):
    # ваш код
    pass

def recall(y_true, y_pred):
    # ваш код
    pass

def specificity(y_true, y_pred):
    # ваш код
    pass

In [ ]:
# Проверка Е2 (ячейка готова)
check('accuracy, письма', lambda: accuracy(y_toy, pred_toy), 0.8)
check('precision, письма', lambda: precision(y_toy, pred_toy), 0.75)
check('recall, письма', lambda: recall(y_toy, pred_toy), 0.75)
check('specificity, письма', lambda: specificity(y_toy, pred_toy), 5 / 6)
check('precision, ленивая модель', lambda: precision(y_spam, pred_lazy), 0.0)
check('accuracy, опухоли', lambda: accuracy(yb_te, predb), float(skm.accuracy_score(yb_te, predb)))
check('precision, опухоли', lambda: precision(yb_te, predb), float(skm.precision_score(yb_te, predb)))
check('recall, опухоли', lambda: recall(yb_te, predb), float(skm.recall_score(yb_te, predb)))

print('\nПарадокс accuracy на спам-фильтре:')
for name, pred in [('честная модель', pred_model), ('ленивая модель', pred_lazy)]:
    try:
        print(f'  {name:15s} accuracy = {accuracy(y_spam, pred):.3f}   recall = {recall(y_spam, pred):.3f}')
    except Exception:
        print('  допишите функции Е2')

**Вопрос Е2.** «Ленивая» модель ничего не делает, но ее accuracy выше, чем у честной (90,9 % против 86,4 %). Почему так вышло?
Какая метрика сразу показывает, что ленивая модель бесполезна? Когда accuracy все-таки можно доверять?

*Ваш ответ:*

### Задание Е3. F-мера: одно число вместо двух

Precision и recall тянут в разные стороны, поэтому их объединяют **средним гармоническим**:

$$F_1 = \frac{2 \cdot P \cdot R}{P + R}, \qquad F_\beta = \frac{(1 + \beta^2)\cdot P \cdot R}{\beta^2 \cdot P + R}$$

- $F_1$ – обе метрики одинаково важны;
- $\beta > 1$ (например, $F_2$) – recall важнее: лучше лишний раз проверить, чем пропустить;
- $\beta < 1$ (например, $F_{0.5}$) – precision важнее: лучше промолчать, чем ошибиться.

Если $P + R = 0$, верните `0.0`. Функции принимают **числа** $P$ и $R$, а не списки.

In [ ]:
# ЗАДАНИЕ Е3
def f_beta(p, r, beta=1.0):
    # ваш код
    pass

def f1(p, r):
    # ваш код (одна строка через f_beta)
    pass

In [ ]:
# Проверка Е3 (ячейка готова)
check('F1(0.75, 0.75)', lambda: f1(0.75, 0.75), 0.75)
check('F1(0.9, 0.5)', lambda: f1(0.9, 0.5), 0.642857)
check('F2(0.9, 0.5)', lambda: f_beta(0.9, 0.5, 2), 0.548780)
check('F0.5(0.9, 0.5)', lambda: f_beta(0.9, 0.5, 0.5), 0.775862)
check('F1(1.0, 0.0)', lambda: f1(1.0, 0.0), 0.0)
check('F1, опухоли', lambda: f1(precision(yb_te, predb), recall(yb_te, predb)), float(skm.f1_score(yb_te, predb)))
print('\nДля сравнения: обычное среднее (0.9 + 0.5) / 2 = 0.7')

**Вопрос Е3.** Для $P = 0{,}9$ и $R = 0{,}5$ обычное среднее равно 0,7, а $F_1 \approx 0{,}64$. Почему среднее гармоническое «наказывает»
перекос? Чему равна $F_1$ у ленивой модели? Какую $F_\beta$ вы бы выбрали для поиска опухолей, а какую – для спам-фильтра?

*Ваш ответ:*

### Задание Е4. Порог классификации

Модель выдает не метку, а **вероятность** $p$. Метку получаем сравнением с порогом $t$:

$$\hat y = \begin{cases} 1, & p \ge t \\ 0, & p < t \end{cases}$$

Порог 0,5 – просто договоренность. Ниже порог – модель чаще говорит «1»: recall растет, precision падает. Выше – наоборот.

1. Напишите `predict_by_threshold(probs, t)` – список меток 0/1.
2. Заполните цикл: для каждого порога из `thresholds` посчитайте precision и recall логистической регрессии на опухолях (`yb_te`, `pb_logreg`).

In [ ]:
# ЗАДАНИЕ Е4
def predict_by_threshold(probs, t):
    # ваш код
    pass

thresholds = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
prec_list, rec_list = [], []
for t in thresholds:
    pred_t = ...                 # <-- метки при пороге t
    prec_list.append(...)        # <-- precision
    rec_list.append(...)         # <-- recall

In [ ]:
# Проверка Е4 и график (ячейка готова)
check('порог 0.5 = pred_toy', lambda: predict_by_threshold(p_toy, 0.5), pred_toy)
check('порог 0.35', lambda: predict_by_threshold(p_toy, 0.35), [1, 1, 1, 0, 1, 1, 1, 0, 0, 0])
for t, pr, rc in zip(thresholds, prec_list, rec_list):
    print(f'порог {t:.2f}   precision {pr:.3f}   recall {rc:.3f}')

plt.figure(figsize=(8, 4))
plt.plot(thresholds, prec_list, 'o-', label='precision')
plt.plot(thresholds, rec_list, 's-', label='recall')
plt.xlabel('порог t'); plt.ylabel('значение метрики'); plt.grid(alpha=.3); plt.legend()
plt.title('Порог меняет баланс precision и recall')
plt.show()

**Вопрос Е4.** Какой порог вы бы выбрали для **первичного** обследования (важно не пропустить больного) и почему?
Какой – если после «положительного» ответа пациента ждет сложная и дорогая операция?

*Ваш ответ:*

### Задание Е5. ROC-кривая и AUC

Чтобы оценить модель **сразу при всех порогах**, строят ROC-кривую – точки с координатами

$$\text{FPR} = \frac{FP}{FP + TN} \;(\text{доля ложных тревог}), \qquad \text{TPR} = \frac{TP}{TP + FN} \;(\text{это recall}).$$

**Как построить.**
1. Начните с точки $(0, 0)$ – порог выше всех вероятностей, модель всем говорит «0».
2. Возьмите пороги – все **уникальные** вероятности по убыванию: `sorted(set(scores), reverse=True)`.
3. Для каждого порога получите метки (Е4), матрицу ошибок (Е1) и добавьте точку (FPR, TPR).
   Последний порог – самая маленькая вероятность, модель всем говорит «1», точка $(1, 1)$.

**AUC** – площадь под кривой. Считаем ее **трапециями** между соседними точками:

$$\text{AUC} = \sum_{i} \left(x_{i+1} - x_i\right)\cdot\frac{y_i + y_{i+1}}{2}$$

AUC = 1 – идеальная модель, AUC = 0,5 – случайное угадывание (диагональ).

In [ ]:
# ЗАДАНИЕ Е5
def roc_curve(y_true, scores):
    fpr, tpr = [0.0], [0.0]
    # ваш код: цикл по порогам, для каждого – одна точка
    return fpr, tpr

def auc(x, y):
    # ваш код: сумма площадей трапеций
    pass

In [ ]:
# Проверка Е5 и график (ячейка готова)
check('площадь квадрата', lambda: auc([0, 1], [1, 1]), 1.0)
check('площадь под диагональю', lambda: auc([0, 1], [0, 1]), 0.5)
check('AUC, письма', lambda: auc(*roc_curve(y_toy, p_toy)), 20 / 24)
check('AUC, логрегрессия', lambda: auc(*roc_curve(yb_te, pb_logreg)), float(skm.roc_auc_score(yb_te, pb_logreg)))
check('AUC, дерево глубины 2', lambda: auc(*roc_curve(yb_te, pb_tree)), float(skm.roc_auc_score(yb_te, pb_tree)))

try:
    plt.figure(figsize=(5.5, 5.5))
    for name, scores in [('логистическая регрессия', pb_logreg), ('дерево глубины 2', pb_tree)]:
        fx, fy = roc_curve(yb_te, scores)
        plt.plot(fx, fy, '.-', label=f'{name}, AUC = {auc(fx, fy):.3f}')
    fx, fy = roc_curve(y_toy, p_toy)
    plt.plot(fx, fy, 'o--', label=f'письма, AUC = {auc(fx, fy):.3f}')
    plt.plot([0, 1], [0, 1], ':', c='gray', label='случайное угадывание')
    plt.xlabel('FPR – доля ложных тревог'); plt.ylabel('TPR – recall'); plt.grid(alpha=.3); plt.legend()
    plt.title('ROC-кривые'); plt.show()
except Exception as e:
    print('Допишите функции Е5:', e)

**Вопрос Е5.** У писем AUC = 20/24. Проверьте другое толкование AUC: возьмите **все пары** «спамное письмо, обычное письмо»
(их 4 · 6 = 24) и посчитайте, в скольких парах модель дала спаму **большую** вероятность. Совпало? Что тогда означает AUC = 0,5?
У дерева глубины 2 на ROC-кривой всего несколько точек – почему?

*Ваш ответ:*

### Задание Е6. Функции потерь для вероятностей

Метрики выше смотрят только на метки. Функции потерь проверяют саму **уверенность** модели: чем меньше, тем лучше.

**Log loss** (логистическая функция потерь, на ней обучается логистическая регрессия):

$$\text{LogLoss} = -\frac{1}{n}\sum_{i=1}^{n}\Big(y_i \ln p_i + (1 - y_i)\ln(1 - p_i)\Big)$$

Для объекта класса 1 штраф равен $-\ln p$: при $p = 0{,}9$ – всего 0,11, а при $p = 0{,}1$ – уже 2,3.
Уверенная ошибка стоит очень дорого. Чтобы не взять $\ln 0$, зажмите $p$ в интервал $[\varepsilon,\; 1-\varepsilon]$, $\varepsilon = 10^{-15}$.

**Оценка Брайера** – средний квадрат ошибки вероятности:

$$\text{Brier} = \frac{1}{n}\sum_{i=1}^{n}(p_i - y_i)^2$$

In [ ]:
# ЗАДАНИЕ Е6
def log_loss(y_true, probs, eps=1e-15):
    # ваш код
    pass

def brier(y_true, probs):
    # ваш код
    pass

In [ ]:
# Проверка Е6 (ячейка готова)
check('неуверенно: y=1, p=0.5', lambda: log_loss([1], [0.5]), 0.693147)
check('уверенно и верно: p=0.9', lambda: log_loss([1], [0.9]), 0.105361)
check('уверенно и НЕверно: p=0.1', lambda: log_loss([1], [0.1]), 2.302585)
check('p = 0 не ломает функцию', lambda: round(log_loss([1], [0.0]), 2), 34.54)
check('log loss, письма', lambda: log_loss(y_toy, p_toy), 0.456868)
check('Brier, письма', lambda: brier(y_toy, p_toy), 0.1525)
check('log loss, опухоли', lambda: log_loss(yb_te, pb_logreg), float(skm.log_loss(yb_te, pb_logreg)))
check('Brier, опухоли', lambda: brier(yb_te, pb_logreg), float(skm.brier_score_loss(yb_te, pb_logreg)))

ps = [i / 100 for i in range(1, 100)]
plt.figure(figsize=(7, 3.5))
plt.plot(ps, [-math.log(p) for p in ps], label='log loss: $-\\ln p$')
plt.plot(ps, [(1 - p) ** 2 for p in ps], label='Brier: $(1 - p)^2$')
plt.xlabel('вероятность, которую модель дала верному классу 1'); plt.ylabel('штраф'); plt.grid(alpha=.3); plt.legend()
plt.title('Штраф за один объект класса 1'); plt.show()

**Вопрос Е6.** Сравните log loss логистической регрессии и дерева глубины 2 на опухолях (посчитайте для `pb_tree`). Что будет,
если в листе дерева все объекты одного класса и модель выдала вероятность ровно 0 для объекта класса 1? Почему в log loss
нужен $\varepsilon$, а в Brier – нет?

*Ваш ответ:*

### Задание Е7. Когда классов больше двух

Матрица ошибок становится таблицей $K \times K$: строка – правда, столбец – ответ модели, на диагонали – верные ответы.
Для каждого класса $k$ смотрим «класс $k$ против остальных»:

$$TP_k = M[k][k], \quad FP_k = \sum_i M[i][k] - TP_k \;(\text{столбец}), \quad FN_k = \sum_j M[k][j] - TP_k \;(\text{строка})$$

Потом метрики классов усредняют:
- **macro** – обычное среднее по классам: $\;\text{macro-}F_1 = \frac{1}{K}\sum_k F_{1,k}$ (все классы равноправны);
- **micro** – сначала складываем $TP_k$, $FP_k$, $FN_k$ всех классов, потом считаем одну метрику. Для precision и recall это просто accuracy.

1. `confusion_matrix_multi(y_true, y_pred, k)` – список списков $k \times k$;
2. `per_class(M)` – список словарей `{'precision': ..., 'recall': ..., 'f1': ...}`, по одному на класс;
3. `macro_f1(M)` и `accuracy_multi(M)` – сумма диагонали, деленная на сумму всех клеток.

In [ ]:
# ЗАДАНИЕ Е7
def confusion_matrix_multi(y_true, y_pred, k):
    M = [[0] * k for _ in range(k)]
    # ваш код
    return M

def per_class(M):
    # ваш код
    pass

def macro_f1(M):
    # ваш код
    pass

def accuracy_multi(M):
    # ваш код
    pass

In [ ]:
# Проверка Е7 и отчет (ячейка готова)
M_small = confusion_matrix_multi([0, 0, 1, 1, 2, 2], [0, 1, 1, 1, 2, 0], 3)
check('маленькая матрица', lambda: M_small, [[1, 1, 0], [0, 2, 0], [1, 0, 1]])
check('accuracy маленькой', lambda: accuracy_multi(M_small), 4 / 6)
Mw = confusion_matrix_multi(yw_te, predw, 3)
check('матрица Wine vs sklearn', lambda: Mw, skm.confusion_matrix(yw_te, predw).tolist())
check('macro-F1 Wine', lambda: macro_f1(Mw), float(skm.f1_score(yw_te, predw, average='macro')))
check('accuracy Wine', lambda: accuracy_multi(Mw), float(skm.accuracy_score(yw_te, predw)))

try:
    print('\nМатрица ошибок Wine (строка – правда, столбец – ответ):')
    for i, row in enumerate(Mw):
        print(f'  сорт {i}:', row)
    print('\n         precision  recall   f1')
    for i, s in enumerate(per_class(Mw)):
        print(f'  сорт {i}   {s["precision"]:.3f}    {s["recall"]:.3f}   {s["f1"]:.3f}')
    print(f'\n  macro-F1 = {macro_f1(Mw):.3f}   accuracy (= micro) = {accuracy_multi(Mw):.3f}')
except Exception as e:
    print('Допишите функции Е7:', e)

**Вопрос Е7.** Какой сорт вина дерево путает чаще всего и с каким? Представьте, что одного класса в 100 раз меньше, чем остальных,
и модель его никогда не угадывает. Какое усреднение – macro или micro – это заметит, а какое почти нет?

*Ваш ответ:*

### Задание Е8. Метрики регрессии

Если модель предсказывает **число**, ошибка на объекте – это разность $y_i - \hat y_i$.

| Метрика | Формула | Смысл |
|---|---|---|
| **MAE** | $\frac{1}{n}\sum \lvert y_i - \hat y_i \rvert$ | средняя ошибка в тех же единицах (°C) |
| **MSE** | $\frac{1}{n}\sum (y_i - \hat y_i)^2$ | средний квадрат ошибки, сильно штрафует большие промахи |
| **RMSE** | $\sqrt{\text{MSE}}$ | MSE, возвращенная в исходные единицы |
| **$R^2$** | $1 - \dfrac{\sum (y_i - \hat y_i)^2}{\sum (y_i - \bar y)^2}$ | насколько модель лучше прогноза «всегда среднее» |

$R^2 = 1$ – идеальный прогноз, $R^2 = 0$ – не лучше среднего $\bar y$, $R^2 < 0$ – хуже среднего.

In [ ]:
# ЗАДАНИЕ Е8
def mae(y_true, y_pred):
    # ваш код
    pass

def mse(y_true, y_pred):
    # ваш код
    pass

def rmse(y_true, y_pred):
    # ваш код
    pass

def r2(y_true, y_pred):
    # ваш код
    pass

In [ ]:
# Проверка Е8 (ячейка готова)
check('MAE, температура', lambda: mae(t_true, t_pred), 0.8)
check('MSE, температура', lambda: mse(t_true, t_pred), 1.1)
check('RMSE, температура', lambda: rmse(t_true, t_pred), math.sqrt(1.1))
check('R2, температура', lambda: r2(t_true, t_pred), 1 - 5.5 / 26)
check('R2 прогноза «среднее»', lambda: r2(t_true, [5] * 5), 0.0)
check('MAE, diabetes', lambda: mae(yd_te, predd), float(skm.mean_absolute_error(yd_te, predd)))
check('MSE, diabetes', lambda: mse(yd_te, predd), float(skm.mean_squared_error(yd_te, predd)), tol=1e-2)
check('R2, diabetes', lambda: r2(yd_te, predd), float(skm.r2_score(yd_te, predd)))

**Вопрос Е8.** В прогнозе температуры одна ошибка в 2 °C, остальные – не больше 1 °C. Какая доля MSE приходится на эту одну ошибку?
Добавьте шестой день с промахом в 10 °C: во сколько раз вырастут MAE и MSE? Какую метрику выбрать, если редкие большие промахи
особенно опасны?

*Ваш ответ:*

---
### Шпаргалка: какую метрику выбрать

| Ситуация | Метрика |
|---|---|
| Классы примерно поровну, ошибки одинаково плохи | accuracy |
| Пропуск страшнее ложной тревоги (болезнь, пешеход перед машиной) | recall, $F_2$ |
| Ложная тревога дороже пропуска (спам, дорогая проверка) | precision, $F_{0.5}$ |
| Нужно одно число и важны обе ошибки | $F_1$ |
| Классов много, редкие важны не меньше частых | macro-$F_1$ |
| Порог еще не выбран, сравниваем модели в целом | ROC-AUC |
| Важны сами вероятности, а не только метки | log loss, Brier |
| Регрессия | MAE (понятно), RMSE (штраф за большие промахи), $R^2$ (сравнение со средним) |

### Задания повышенной сложности (к части 4Е)

1. **Balanced accuracy и G-среднее:** $\frac{\text{recall} + \text{specificity}}{2}$ и $\sqrt{\text{recall}\cdot\text{specificity}}$.
   Сравните их с accuracy на спам-фильтре для честной и ленивой моделей.
2. **Лучший порог.** Переберите пороги от 0,01 до 0,99 и найдите тот, при котором $F_1$ на опухолях максимальна. Совпадает ли он с 0,5?
3. **AUC через пары.** Напишите `auc_pairs(y_true, scores)`: доля пар (положительный, отрицательный), где у положительного
   вероятность больше; ничья дает 0,5. Сверьте с `auc(*roc_curve(...))`. Сколько операций нужно каждой из функций для $n$ объектов?
4. **Average Precision:** $\text{AP} = \sum_n (R_n - R_{n-1})\,P_n$ по тем же порогам, что и в ROC. Сверьте с `skm.average_precision_score`.
5. **MAPE:** $\frac{1}{n}\sum \frac{\lvert y_i - \hat y_i\rvert}{\lvert y_i\rvert}$. Почему ее нельзя считать для температуры, если один из дней был 0 °C?
6. **Свое дерево.** Подставьте в Е7 предсказания дерева, написанного в части 4Д, и сравните macro-$F_1$ с деревом sklearn.